# オートエンコーダ（AE）Functional API編

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARIM-ACADEMY-2026/Advanced_Tutorial_3_Keras/blob/main/3_Keras_AE_Functional-API.ipynb)

`3_Keras_AE_Sequential-API.ipynb`と同じ課題（MNISTの784次元画像を2次元に圧縮・復元する）を、Functional APIで実装します。

## 対象読者・前提知識・動作環境・版とライセンス

- **対象読者**: `3_Keras_AE_Sequential-API.ipynb`と`2_Keras_CNN_Functional-API.ipynb`を読んだ方
- **前提知識**: オートエンコーダの目的（教師なし学習で画像を圧縮・復元する）はSequential編で説明済みとして進める
- **動作環境**: Python 3.10以降、TensorFlow 2.16以降・Keras 3系（`tensorflow.keras`は現在Keras 3を指す）、scikit-learn
- **データセット**: MNIST（手書き数字、Sequential編と共通）
- **版**: 2026-08-03作成

## 目次

1. データの準備（Sequential編と共通）
2. モデルの構築（Functional API）
3. 学習
4. 評価（潜在空間・再構成画像）
5. PCA・Sequential編との比較

## 教材への接続（Google Colab）

Google Colabで開いた場合は、次のセルを実行してこのリポジトリをクローンし、`module/`（共通ヘルパー関数）や`output/`・`comparison_output/`（他ノートブックとの受け渡しファイル）を含むフォルダへ移動してください。ローカル環境でこのフォルダを直接開いている場合は、次のセルは実行不要です（すでにカレントディレクトリが正しい場所になっています）。


In [ ]:
!git clone https://github.com/ARIM-ACADEMY-2026/Advanced_Tutorial_3_Keras.git
%cd Advanced_Tutorial_3_Keras


## 1. データの準備

### ステップ1: ライブラリを読み込み、乱数シードを固定する

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
sys.path.append(str(BASE_DIR))  # module/ はBASE_DIR直下にあるため、BASE_DIR自体をsys.pathに加える

from module.seed_utils import set_seed
from module import data_utils, viz_utils

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

set_seed(42)

### ステップ2: データセットを読み込む（Sequential編と同じ前処理）

In [ ]:
data = data_utils.load_mnist_flat()

train_images, test_images = data.train_images, data.test_images
train_labels, test_labels = data.train_labels, data.test_labels

train_images.shape, test_images.shape

## 2. モデルの構築

### ステップ1: エンコーダ部分を組み立てる

Sequential編と同じ`784 → 512 → 128 → 2`という圧縮の流れを、Functional APIで書きます。各層の出力を`e1`, `encoded`, `coded`という変数に受けていく点は、CNN Functional編と同じ考え方です。

In [ ]:
from tensorflow import keras
from keras.layers import Input, Dense

inputs = Input(shape=(784,))

e1 = Dense(512, activation="relu")(inputs)
encoded = Dense(128, activation="relu")(e1)
coded = Dense(2, activation="linear", name="code-layer")(encoded)

coded.shape

### ステップ2: デコーダ部分を組み立てる

In [ ]:
d1 = Dense(128, activation="relu")(coded)
decoded = Dense(512, activation="relu")(d1)
outputs = Dense(784, activation="sigmoid")(decoded)

outputs.shape

### ステップ3: 「オートエンコーダ全体」と「エンコーダだけ」を、同じ層から2つのモデルとして作る

ここがFunctional APIらしい書き方です。`coded`という同じテンソル（層のつながり）を使い回して、2つの異なる`Model`を定義できます。

- `autoencoder`: `inputs`から`outputs`まで（圧縮して復元する、全体のモデル）
- `encoder_func`: `inputs`から`coded`まで（圧縮するところまでの、部分的なモデル）

Sequential編では、学習が終わったあとに`model.get_layer("code-layer")`で後追いでエンコーダ部分を取り出しました。Functional APIでは、モデルを定義する時点で最初から両方のモデルを用意しておける、という違いがあります。

In [ ]:
autoencoder = keras.Model(inputs=inputs, outputs=outputs, name="autoencoder")
encoder_func = keras.Model(inputs=inputs, outputs=coded, name="encoder")

autoencoder.summary()

In [ ]:
encoder_func.summary()

`autoencoder`と`encoder_func`は、`Dense(512)`・`Dense(128)`・`code-layer`という同じ層のインスタンスを共有しています。つまり、後で`autoencoder`を学習させると、`encoder_func`の重みも（同じ層を参照しているため）自動的に更新されます。層とモデルが別概念であることを利用した、Functional APIらしいテクニックです。

**コラム: それでも学習後に`get_layer`で取り出し直す理由**

`encoder_func`は`autoencoder`と重みを共有しているので、理屈の上では学習後もそのまま使えるはずです。ただし本ノートブックでは、Sequential編と手順をそろえるため、3節で学習後に改めて`get_layer("code-layer")`から作り直します。同じ層を共有しているモデルが複数存在するとき、「今どの時点の重みを参照しているか」を明示的に確認する習慣をつけておくと安全です。

### ステップ4: モデルをコンパイルする

In [ ]:
autoencoder.compile(optimizer="adam", loss="mean_squared_error")

## 3. 学習

In [ ]:
%%time
history = autoencoder.fit(
    train_images,
    train_images,
    batch_size=256,
    epochs=30,
    verbose=1,
    validation_data=(test_images, test_images),
)

## 4. 評価（潜在空間・再構成画像）

### ステップ1: 学習曲線を確認する（図1）

In [ ]:
viz_utils.plot_metric_curve(history, metric="loss", fig_num=1, output_dir=OUTPUT_DIR)

### ステップ2: 学習後のエンコーダを取り出す

2節のコラムで触れたとおり、`get_layer("code-layer")`を使って、学習済みの`autoencoder`から改めてエンコーダを作り直します。

In [ ]:
encoder_trained = keras.Model(
    inputs=autoencoder.input,
    outputs=autoencoder.get_layer("code-layer").output,
    name="encoder",
)

latent_train_ae_func = encoder_trained.predict(train_images, verbose=0)
latent_train_ae_func.shape

### ステップ3: 潜在空間を可視化する（図2）

In [ ]:
viz_utils.plot_latent_scatter(
    latent_train_ae_func, train_labels,
    class_names=data_utils.MNIST_CLASS_NAMES, fig_num=2, output_dir=OUTPUT_DIR,
)

### ステップ4: 再構成画像を確認する（図3）

In [ ]:
reconstruction_train_ae_func = autoencoder.predict(train_images, verbose=0)

viz_utils.plot_reconstruction_grid(
    train_images, reconstruction_train_ae_func,
    n=10, image_shape=(28, 28), fig_num=3, output_dir=OUTPUT_DIR,
)

### ステップ5: 学習済みモデルと結果を保存する

In [ ]:
train_mse_ae_func = float(np.mean((train_images - reconstruction_train_ae_func) ** 2))
print("訓練データに対する再構成MSE:", train_mse_ae_func)

autoencoder.save(OUTPUT_DIR / "ae_func_trained.keras")
np.savez(
    OUTPUT_DIR / "ae_func_results.npz",
    latent_train=latent_train_ae_func,
    reconstruction_train=reconstruction_train_ae_func,
    train_mse=train_mse_ae_func,
)
print(f"保存先: {OUTPUT_DIR / 'ae_func_results.npz'}")

## 5. PCA・Sequential編との比較

### ステップ1: PCAで784次元を2次元に圧縮・復元する

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
latent_train_pca = pca.fit_transform(train_images)
reconstruction_train_pca = pca.inverse_transform(latent_train_pca)

train_mse_pca = float(np.mean((train_images - reconstruction_train_pca) ** 2))
print("PCAの再構成MSE:", train_mse_pca)

### ステップ2: AE(Functional) と PCA を並べて比較する（図4）

In [ ]:
viz_utils.plot_reconstruction_grid(
    train_images,
    {"AE (Functional)": reconstruction_train_ae_func, "PCA": reconstruction_train_pca},
    n=10, image_shape=(28, 28), fig_num=4, output_dir=OUTPUT_DIR,
)

### ステップ3: Sequential編の結果があれば読み込んで3者を比較する

In [ ]:
seq_results_path = BASE_DIR / "output" / "ae_seq_results.npz"

if seq_results_path.exists():
    seq_results = np.load(seq_results_path)
    train_mse_ae_seq = float(seq_results["train_mse"])
    print(f"AE (Sequential) の再構成MSE: {train_mse_ae_seq:.5f}")
else:
    train_mse_ae_seq = None
    print("Sequential編の output/ae_seq_results.npz が見つからないため、比較をスキップしました。"
          " 3_Keras_AE_Sequential-API.ipynb を先に実行してください。")

print(f"AE (Functional)  の再構成MSE: {train_mse_ae_func:.5f}")
print(f"PCA              の再構成MSE: {train_mse_pca:.5f}")

AE(Sequential)とAE(Functional)は、同じ構成・同じデータ・同じ乱数シードで学習させているため、MSEはほぼ一致するはずです（完全な再現性はGPU実行時には保証されない点に注意、詳しくはMLP編1節ステップ1のコラムを参照）。書き方（API）による差ではなく、モデルの構成そのものの差でPCAとの違いが生まれていることを確認してください。

### ステップ4: 比較表に記録する

In [ ]:
import csv
import pandas as pd

COMPARISON_DIR = BASE_DIR.parent / "comparison_output"
COMPARISON_DIR.mkdir(exist_ok=True)
comparison_path = COMPARISON_DIR / "reconstruction_comparison.csv"

rows = [{"method": "AE", "api_style": "Functional", "dataset": "MNIST", "latent_dim": 2, "train_mse": train_mse_ae_func}]

# PCAの行は、Sequential編ですでに記録済みなら重複させない
already_has_pca = False
if comparison_path.exists():
    existing_df = pd.read_csv(comparison_path)
    already_has_pca = (existing_df["method"] == "PCA").any()
if not already_has_pca:
    rows.append({"method": "PCA", "api_style": "-", "dataset": "MNIST", "latent_dim": 2, "train_mse": train_mse_pca})

file_exists = comparison_path.exists()
with open(comparison_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["method", "api_style", "dataset", "latent_dim", "train_mse"])
    if not file_exists:
        writer.writeheader()
    writer.writerows(rows)

pd.read_csv(comparison_path)

## まとめ

- Sequential編と同じオートエンコーダを、Functional APIで組み立てた。`inputs`から`coded`まで、`inputs`から`outputs`までの2つの`Model`を同じ層から定義できる点がFunctional APIらしい書き方だった
- 学習後は`get_layer`で重み更新済みのエンコーダを取り出す必要があり、この点はSequential編と共通の手順になることを確認した
- PCA・AE(Sequential)・AE(Functional)の3手法を再構成MSEで比較した

## 本ノートブックで扱っていないこと（今後の課題）

- 潜在空間の次元数を増やした場合の精度変化（Sequential編と共通の課題）
- 「なめらかに意味が変化する潜在空間」を明示的に学習する変分オートエンコーダ（VAE）。次の`2_Keras_VAE_Functional-API.ipynb`で扱う

## 演習問題

1. `encoder_func`（2節で作った学習前のエンコーダ）と`encoder_trained`（4節で学習後に作り直したエンコーダ）に同じ画像を入力し、出力する潜在表現が異なることを確認しなさい（学習前後で重みが変わったことの確認）。
2. `autoencoder.get_layer("code-layer")`を使わずに、`coded`という変数（2節でモデル定義時に作った、層のつながりを表すテンソル）を直接使ってエンコーダを再定義できるか考えなさい（両者は同じ結果になるはずです。同じKernel内で変数が生き続けているため、どちらの書き方でも動く点を確認してみましょう）。
3. PCA・AE(Sequential)・AE(Functional)の3手法の再構成MSEを棒グラフにまとめなさい。